# Colab runner

Launcher only — no method code lives here. See `model.py`.

**Connect first:** `Select Kernel` → `Colab` → `New Colab Server` → pick **GPU**.
Then run these cells top to bottom.

Everything below executes on the Colab VM, not on your laptop.


## 1. Confirm we actually got a GPU

In [1]:
!nvidia-smi
import tensorflow as tf
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

Sun Sep 20 03:06:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   47C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Drive

(Or use the command palette: `Colab: Mount Google Drive to Server...`)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Clone the repo onto the VM

`/content` is wiped when the runtime dies, so this runs every fresh session.

In [ ]:
%cd /content
!git clone https://github.com/Dev-Joyson/CNN-based-GAN-face-detection.git 2>/dev/null || (cd CNN-based-GAN-face-detection && git pull)
%cd /content/CNN-based-GAN-face-detection
!pip install -q pyyaml


## 4. Sanity check: do the guard tests pass on this machine?

~3 seconds, no dataset needed.

In [4]:
!pytest tests -q

............                                                             [100%]
12 passed in 10.99s


## 5. Restore the crop cache from Drive

The headline pipeline caches 512² native crops (~33 GB). On a fresh VM this copy
takes ~20 min; without it, epoch 1 re-decodes 50k PNGs off Drive (much slower).


In [ ]:
!python cache_sync.py --config configs/test19_sg2_crop.yaml --from-drive


## 6. Headline: native-resolution crops, no FFT branch (test19)

**Run long jobs from the VM terminal, not this cell** (`Cmd+Shift+P` → `Colab: Open
Terminal`), so they survive the notebook frontend dying:

```
nohup python -u train.py --config configs/test19_sg2_crop_no_fft.yaml > /content/train.log 2>&1 &
tail -f /content/train.log
```

Add `--resume` after a VM reclaim. The cell below is the same thing, for a quick test only.


In [ ]:
!python train.py --config configs/test19_sg2_crop_no_fft.yaml


## 7. Ablation and seeds

FFT branch on the same crops (`test19_sg2_crop`), and seeds 43 / 44 of the headline.
Same `nohup` pattern from the terminal.


In [ ]:
!python train.py --config configs/test19_sg2_crop.yaml
!python train.py --config configs/test19_sg2_crop_no_fft_s43.yaml
!python train.py --config configs/test19_sg2_crop_no_fft_s44.yaml


## 8. Evaluate, baselines, held-out generator

`evaluate.py` writes `eval.json`, confusion matrix, ROC, Grad-CAM and the efficiency
block (params, MACs, GPU/CPU latency, memory) into the run folder on Drive.

Baselines fine-tune on the **same crop pipeline** (accuracy column only; their
efficiency numbers are already final). `heldout.py` scores a trained model on
StyleGAN3-T images it never saw.


In [ ]:
!python evaluate.py --config configs/test19_sg2_crop_no_fft.yaml
!python evaluate.py --config configs/test19_sg2_crop.yaml


In [ ]:
# one at a time, from the terminal under nohup: mobilenet_v3_small -> efficientnet_b0 -> xception
!python baselines.py --config configs/test19_sg2_crop_no_fft.yaml --model mobilenet_v3_small --train


In [ ]:
!python heldout.py --config configs/test19_sg2_crop_no_fft.yaml --fake-dir "/content/drive/MyDrive/Fake(SG3-T-psi1)" --tag sg3t
!python heldout.py --config configs/test19_sg2_crop.yaml --fake-dir "/content/drive/MyDrive/Fake(SG3-T-psi1)" --tag sg3t


## 9. Watch training

**Live:** TensorBoard reads `<run>/tb/` and refreshes itself every 30 s.
Pointing it at the experiments root shows every run on one chart.
If the panel does not render inside VS Code, open this same notebook at
colab.research.google.com — it embeds there.

**Snapshot:** the cell after reads `history.csv` from Drive; re-run it any time.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/Research/experiments"


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
h = pd.read_csv("/content/drive/MyDrive/Research/experiments/test19_sg2_crop_no_fft/history.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
h[["accuracy", "val_accuracy"]].plot(ax=ax[0], title="accuracy")
h[["auc", "val_auc"]].plot(ax=ax[1], title="AUC")
plt.tight_layout(); plt.show()

In [ ]:
import time, datetime
for _ in range(100000):
    print(datetime.datetime.now().strftime("%H:%M:%S"), "alive", flush=True)
    time.sleep(60)